# SetFit Inference - Phase 1 (GPU Accelerated)

**Purpose**: Run SetFit inference on 20,816 medium-score papers using trained model

**Estimated time**: ~6 minutes on GPU (vs 20+ minutes on CPU)

**Steps**:
1. Install dependencies
2. Mount Google Drive
3. Load trained SetFit model from Drive
4. Load 20,816 medium-score papers
5. Run inference on GPU
6. Save results back to Drive

## 1. Setup Environment

In [ ]:
# Install dependencies
!pip install -q setfit datasets pandas numpy

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Imports
import os
import time
import pandas as pd
import numpy as np
from datetime import datetime
from setfit import SetFitModel

print("✓ Imports complete")

## 2. Define Paths

In [ ]:
# Google Drive paths
base_dir = "/content/drive/MyDrive/inventory_2022"
model_dir = f"{base_dir}/advanced_paper_filtering/models/2025-11-17-134146_setfit_training/setfit_introduction_classifier"
data_dir = f"{base_dir}/advanced_paper_filtering/data"
output_dir = f"{base_dir}/pipeline_synthesis_2025-11-18/results/setfit_inference"

# Create output directory
os.makedirs(output_dir, exist_ok=True)

print(f"Model directory: {model_dir}")
print(f"Data directory: {data_dir}")
print(f"Output directory: {output_dir}")

## 3. Load Trained Model

In [ ]:
print("Loading SetFit model from Drive...")
print(f"Path: {model_dir}")

model = SetFitModel.from_pretrained(model_dir)

print("✓ Model loaded successfully")
print(f"Device: {model.model_body.device}")

## 4. Load Medium-Score Papers

In [ ]:
# Load medium-score papers
papers_file = f"{data_dir}/results/medium_score_papers.csv"

print(f"Loading papers from: {papers_file}")
df = pd.read_csv(papers_file)

print(f"✓ Loaded {len(df):,} papers")
print(f"Columns: {list(df.columns)}")

## 5. Prepare Text Inputs

In [ ]:
print("Preparing text inputs...")

texts = []
for idx, row in df.iterrows():
    title = str(row['title']) if pd.notna(row['title']) else ""
    abstract = str(row['abstract']) if pd.notna(row['abstract']) else ""
    
    if abstract:
        text = f"{title}\n\n{abstract}"
    else:
        text = title
    
    texts.append(text)

print(f"✓ Prepared {len(texts):,} text inputs")
print(f"\nSample text (first 200 chars):")
print(texts[0][:200])

## 6. Run Inference (GPU Accelerated)

In [ ]:
print(f"Running inference on {len(texts):,} papers...")
print(f"Device: {model.model_body.device}")
print("Estimated time: ~6 minutes on GPU")
print()

start_time = time.time()

# Get predictions and probabilities
predictions = model.predict(texts)
proba = model.predict_proba(texts)

# Extract confidence scores (probability of positive class)
if len(proba.shape) == 2:
    confidence = proba[:, 1]
else:
    confidence = proba

elapsed = time.time() - start_time

print(f"✓ Inference complete in {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")
print(f"  Average: {elapsed/len(texts):.3f} sec/paper")

## 7. Analyze Results

In [ ]:
print("Categorizing by confidence tier...\n")

# Define tiers
high_mask = confidence >= 0.70
medium_mask = (confidence >= 0.60) & (confidence < 0.70)
low_mask = confidence < 0.60

# Count by tier and prediction
intro_high = sum(predictions & high_mask)
intro_medium = sum(predictions & medium_mask)
intro_low = sum(predictions & low_mask)

usage_high = sum((~predictions) & high_mask)
usage_medium = sum((~predictions) & medium_mask)
usage_low = sum((~predictions) & low_mask)

print("Introductions:")
print(f"  High confidence (≥0.70):    {intro_high:,} ({intro_high/len(predictions)*100:.1f}%)")
print(f"  Medium confidence (0.60-0.69): {intro_medium:,} ({intro_medium/len(predictions)*100:.1f}%)")
print(f"  Low confidence (<0.60):     {intro_low:,} ({intro_low/len(predictions)*100:.1f}%)")
print(f"  TOTAL:                      {sum(predictions):,}")

print("\nUsage:")
print(f"  High confidence (≥0.70):    {usage_high:,} ({usage_high/len(predictions)*100:.1f}%)")
print(f"  Medium confidence (0.60-0.69): {usage_medium:,} ({usage_medium/len(predictions)*100:.1f}%)")
print(f"  Low confidence (<0.60):     {usage_low:,} ({usage_low/len(predictions)*100:.1f}%)")
print(f"  TOTAL:                      {sum(~predictions):,}")

## 8. Save Results

In [ ]:
print(f"Saving results to: {output_dir}\n")

# Add predictions to dataframe
results_df = df.copy()
results_df['setfit_prediction'] = predictions.astype(int)
results_df['setfit_confidence'] = confidence

# Add confidence tier
conditions = [
    confidence >= 0.70,
    (confidence >= 0.60) & (confidence < 0.70),
    confidence < 0.60
]
choices = ['high', 'medium', 'low']
results_df['confidence_tier'] = np.select(conditions, choices, default='unknown')

# Add prediction label
results_df['prediction_label'] = results_df['setfit_prediction'].map({
    1: 'INTRODUCTION',
    0: 'USAGE'
})

# Save introductions
intro_df = results_df[results_df['setfit_prediction'] == 1].copy()
intro_df = intro_df.sort_values('setfit_confidence', ascending=False)
intro_file = f"{output_dir}/setfit_classified_introductions.csv"
intro_df.to_csv(intro_file, index=False)
print(f"✓ Saved {len(intro_df):,} introductions: {intro_file}")

# Save usage
usage_df = results_df[results_df['setfit_prediction'] == 0].copy()
usage_df = usage_df.sort_values('setfit_confidence', ascending=False)
usage_file = f"{output_dir}/setfit_classified_usage.csv"
usage_df.to_csv(usage_file, index=False)
print(f"✓ Saved {len(usage_df):,} usage papers: {usage_file}")

# Save combined results
combined_file = f"{output_dir}/setfit_all_results.csv"
results_df = results_df.sort_values('setfit_confidence', ascending=False)
results_df.to_csv(combined_file, index=False)
print(f"✓ Saved {len(results_df):,} total results: {combined_file}")

## 9. Generate Summary Report

In [ ]:
summary_file = f"{output_dir}/setfit_inference_summary.txt"

with open(summary_file, 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("SETFIT INFERENCE SUMMARY - PHASE 1 (GPU)\n")
    f.write("=" * 80 + "\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Total papers classified: {len(intro_df) + len(usage_df):,}\n")
    f.write(f"Inference time: {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)\n")
    f.write("Device: GPU\n")
    f.write("\n")
    
    f.write("CLASSIFICATION RESULTS\n")
    f.write("-" * 80 + "\n")
    total = len(intro_df) + len(usage_df)
    f.write(f"Introductions: {len(intro_df):,} ({len(intro_df)/total*100:.1f}%)\n")
    f.write(f"Usage:         {len(usage_df):,} ({len(usage_df)/total*100:.1f}%)\n")
    f.write("\n")
    
    f.write("CONFIDENCE TIERS - INTRODUCTIONS\n")
    f.write("-" * 80 + "\n")
    f.write(f"High (≥0.70):       {intro_high:,} ({intro_high/len(intro_df)*100:.1f}% of intros)\n")
    f.write(f"Medium (0.60-0.69): {intro_medium:,} ({intro_medium/len(intro_df)*100:.1f}% of intros)\n")
    f.write(f"Low (<0.60):        {intro_low:,} ({intro_low/len(intro_df)*100:.1f}% of intros)\n")
    f.write("\n")
    
    f.write("CONFIDENCE TIERS - USAGE\n")
    f.write("-" * 80 + "\n")
    f.write(f"High (≥0.70):       {usage_high:,} ({usage_high/len(usage_df)*100:.1f}% of usage)\n")
    f.write(f"Medium (0.60-0.69): {usage_medium:,} ({usage_medium/len(usage_df)*100:.1f}% of usage)\n")
    f.write(f"Low (<0.60):        {usage_low:,} ({usage_low/len(usage_df)*100:.1f}% of usage)\n")
    f.write("\n")
    
    f.write("CONFIDENCE STATISTICS - INTRODUCTIONS\n")
    f.write("-" * 80 + "\n")
    intro_conf = intro_df['setfit_confidence']
    f.write(f"Mean:   {intro_conf.mean():.4f}\n")
    f.write(f"Median: {intro_conf.median():.4f}\n")
    f.write(f"Min:    {intro_conf.min():.4f}\n")
    f.write(f"Max:    {intro_conf.max():.4f}\n")
    f.write(f"Std:    {intro_conf.std():.4f}\n")
    f.write("\n")
    
    f.write("CONFIDENCE STATISTICS - USAGE\n")
    f.write("-" * 80 + "\n")
    usage_conf = usage_df['setfit_confidence']
    f.write(f"Mean:   {usage_conf.mean():.4f}\n")
    f.write(f"Median: {usage_conf.median():.4f}\n")
    f.write(f"Min:    {usage_conf.min():.4f}\n")
    f.write(f"Max:    {usage_conf.max():.4f}\n")
    f.write(f"Std:    {usage_conf.std():.4f}\n")
    f.write("\n")
    f.write("=" * 80 + "\n")

print(f"✓ Summary saved: {summary_file}")

# Print summary
print("\n" + "=" * 80)
print("INFERENCE COMPLETE")
print("=" * 80)
print(f"Total papers: {total:,}")
print(f"Introductions: {len(intro_df):,} ({len(intro_df)/total*100:.1f}%)")
print(f"  High confidence: {intro_high:,}")
print(f"  Medium confidence: {intro_medium:,}")
print(f"Usage: {len(usage_df):,} ({len(usage_df)/total*100:.1f}%)")
print(f"  High confidence: {usage_high:,}")
print(f"  Medium confidence: {usage_medium:,}")
print("=" * 80)
print(f"\n✓ Phase 1 complete!")
print(f"Results saved to: {output_dir}")